<a href="https://colab.research.google.com/github/yjhe99/OF3-with-batch-json-input/blob/main/OpenFold3_JSON_Batch_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold 3 (OpenFold3)

Predict protein, RNA, DNA, and small-molecule structures using [AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) with the freely available [OpenFold3](https://github.com/aqlaboratory/openfold) weights (Apache 2.0 — no commercial restrictions).

MSA generation via the [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server — **no local databases required**.

Just fill in the **Input sequence(s)** cell and run all. Proteins, nucleic acids and ligands all go in a single box; attention/XLA flags are chosen automatically for your runtime (T4, newer GPUs, or CPU).

**Citations:**
- Abramson et al. (2024) AlphaFold 3. *Nature* [doi:10.1038/s41586-024-07487-w](https://doi.org/10.1038/s41586-024-07487-w)
- Mirdita et al. (2022) ColabFold. *Nature Methods* [doi:10.1038/s41592-022-01488-1](https://doi.org/10.1038/s41592-022-01488-1)
- The OpenFold3 Team (2025) [Github](https://github.com/aqlaboratory/openfold-3) - [doi:10.5281/zenodo.19001000](https://doi.org/10.5281/zenodo.19001000)

**Credits:** AF3 code: Google DeepMind (Apache 2.0) · OF3 weights: AlQuraishi Lab (Apache 2.0)
**Batch JSON edition:** accepts AlphaFold Server or native AF3 JSON files, saves to Drive, and resumes completed jobs.


In [ ]:
!pip install --upgrade git+https://github.com/sokrypton/alphafold3.git

In [ ]:
#@title Install dependencies (~3 mins)
%%time
import os, time, glob

use_af3_weights = True #@param {type:"boolean"}
#@markdown - **use_af3_weights**: leave OFF to use the free, open **OpenFold3** weights. Turn ON to
#@markdown   download the official **AlphaFold 3** weights directly - now public, no login or key required.
#@markdown   Subject to the AF3 weights terms of use.
USE_NATIVE = use_af3_weights
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.17 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol")
  os.system("pip install -q --no-deps 'https://github.com/sokrypton/alphafold3/releases/download/v3.1.5/alphafold3_open-3.1.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'")
  base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/refs/heads/main'
  for script in ['run_alphafold', 'convert_of3_weights']:
    os.system(f'wget -q -nc {base}/{script}.py')
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights: official AlphaFold 3 (public download) or OpenFold3 (default, free) - both in background
if USE_NATIVE:
  if not os.path.isfile('NATIVE_WEIGHTS_DONE'):
    print('Downloading official AlphaFold 3 weights (public, no login required)...')
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -q -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"; touch NATIVE_WEIGHTS_DONE) &')
elif not os.path.isfile('WEIGHTS_DONE'):
  print('Downloading OpenFold3 weights...')
  os.system('(aws s3 cp s3://openfold/staging/of3-p2-155k.pt . --no-sign-request; touch WEIGHTS_DONE) &')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

# Wait for background jobs
for sentinel in (['NATIVE_WEIGHTS_DONE', 'DATA_DONE'] if USE_NATIVE else ['WEIGHTS_DONE', 'DATA_DONE']):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if USE_NATIVE and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

if not USE_NATIVE and not os.path.isfile('af3_converted_weights/of3_ported_weights.bin.zst'):
  print("converting openfold3 weights...")
  os.system("python convert_of3_weights.py --of3_checkpoint of3-p2-155k.pt --output_dir af3_converted_weights")

print('Setup complete!  ' + ('Using official AlphaFold 3 weights.' if USE_NATIVE else 'Using OpenFold3 weights.'))

In [ ]:
!wget -q -O run_alphafold.py \
  https://raw.githubusercontent.com/sokrypton/alphafold3/v3.1.5/run_alphafold.py

!wget -q -O convert_of3_weights.py \
  https://raw.githubusercontent.com/sokrypton/alphafold3/v3.1.5/convert_of3_weights.py

In [ ]:
#@title Upload JSON batch(es)
import re, os, json, glob
from pathlib import Path

#@markdown Upload one or more JSON files, or provide a Drive file/directory path.
json_source = "google_drive" #@param ["upload", "google_drive"]
drive_json_path = "/content/mango2chip_lifetime_af3_ins_del_batch_04_fixed_copy.json" #@param {type:"string"}
#@markdown Save every completed job to Drive so interrupted sessions can resume.
save_to_drive = True #@param {type:"boolean"}
drive_output_dir = "/content/drive/MyDrive/openfold3_mango_results/4" #@param {type:"string"}
on_existing = "skip" #@param ["skip", "overwrite"]

if save_to_drive or json_source == 'google_drive':
  from google.colab import drive
  drive.mount('/content/drive', force_remount=False)

if json_source == 'upload':
  from google.colab import files
  uploaded = files.upload()
  json_paths = [Path(name) for name in uploaded]
else:
  p = Path(drive_json_path)
  if p.is_dir():
    json_paths = sorted(p.glob('*.json'))
  elif p.is_file():
    json_paths = [p]
  else:
    raise FileNotFoundError(f'JSON path not found: {p}')

if not json_paths:
  raise ValueError('No JSON files selected.')

CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')

def _server_to_af3(job):
  """Convert AlphaFold Server dialect (including v3 exports) to AF3/OpenFold3."""
  chains, idx = [], 0
  for entity in job.get('sequences', []):
    if idx >= len(CHAIN_IDS):
      raise ValueError(f'Too many entities in {job.get("name", "unnamed job")}')
    cid = CHAIN_IDS[idx]
    if 'rnaSequence' in entity:
      e = entity['rnaSequence']
      count = int(e.get('count', 1))
      ids = CHAIN_IDS[idx:idx+count]
      chain = {'id': ids[0] if count == 1 else ids, 'sequence': e['sequence'].upper()}
      chains.append({'rna': chain}); idx += count
    elif 'dnaSequence' in entity:
      e = entity['dnaSequence']; count = int(e.get('count', 1))
      ids = CHAIN_IDS[idx:idx+count]
      chains.append({'dna': {'id': ids[0] if count == 1 else ids, 'sequence': e['sequence'].upper()}}); idx += count
    elif 'proteinChain' in entity:
      e = entity['proteinChain']; count = int(e.get('count', 1))
      ids = CHAIN_IDS[idx:idx+count]
      chains.append({'protein': {'id': ids[0] if count == 1 else ids, 'sequence': e['sequence'].upper(), 'templates': [], 'unpairedMsa': f">query\\n{e['sequence'].upper()}\\n", 'pairedMsa': ''}}); idx += count
    elif 'ligand' in entity:
      e = entity['ligand']; count = int(e.get('count', 1))
      ids = CHAIN_IDS[idx:idx+count]
      code = e.get('ligand', '').removeprefix('CCD_')
      if not code: raise ValueError(f'Missing ligand code in {job.get("name")}')
      chains.append({'ligand': {'id': ids[0] if count == 1 else ids, 'ccdCodes': [code]}}); idx += count
    elif 'ion' in entity:
      e = entity['ion']; count = int(e.get('count', 1))
      ids = CHAIN_IDS[idx:idx+count]
      chains.append({'ligand': {'id': ids[0] if count == 1 else ids, 'ccdCodes': [e['ion']]}}); idx += count
    else:
      raise ValueError(f'Unsupported entity in {job.get("name")}: {entity.keys()}')
  return {
    'name': job.get('name', 'job'), 'modelSeeds': [int(x) for x in job.get('modelSeeds', [1])],
    'sequences': chains, 'dialect': 'alphafold3', 'version': 1,
  }

def _normalise_job(job):
  if job.get('dialect') == 'alphafoldserver' or any(
      any(k in e for k in ('rnaSequence','dnaSequence','proteinChain','ion'))
      for e in job.get('sequences', [])):
    job = _server_to_af3(job)
  else:
    job = dict(job)
    job['modelSeeds'] = [int(x) for x in job.get('modelSeeds', [1])]
  if not job.get('sequences'):
    raise ValueError(f'No sequences in {job.get("name", "unnamed job")}')
  # This notebook runs with --norun_data_pipeline. Supply a query-only RNA MSA
  # so RNA+ligand jobs remain self-contained and require no external RNA database.
  for entity in job['sequences']:
    if 'rna' in entity and 'unpairedMsa' not in entity['rna']:
      seq = entity['rna']['sequence']
      entity['rna']['unpairedMsa'] = f'>query\n{seq}\n'
  return job

fold_inputs=[]
for path in json_paths:
  data=json.loads(path.read_text())
  if isinstance(data, dict): data=[data]
  if not isinstance(data, list): raise TypeError(f'{path}: top-level JSON must be an object or list')
  fold_inputs.extend(_normalise_job(x) for x in data)

# Unique, filesystem-safe job names while retaining mutant labels.
seen=set()
for n, job in enumerate(fold_inputs, 1):
  base=(re.sub(r'[^A-Za-z0-9_.-]+','_',str(job.get('name') or f'job_{n}')).strip('._') or f'job_{n}')
  name=base; suffix=2
  while name.lower() in seen:
    name=f'{base}_{suffix}'; suffix += 1
  seen.add(name.lower()); job['name']=name

INPUT_DIR='/tmp/af3_batch_inputs'
OUTPUT_DIR=drive_output_dir if save_to_drive else '/content/af3_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
for job in fold_inputs:
  with open(f'{INPUT_DIR}/{job["name"]}.json','w') as f: json.dump(job,f,indent=2)

print(f'Loaded {len(fold_inputs)} jobs from {len(json_paths)} JSON file(s).')
print(f'Persistent output: {OUTPUT_DIR}')
print('First job:', fold_inputs[0]['name'], fold_inputs[0]['sequences'])


In [ ]:
#@title Run JSON batch with OpenFold3
%%time
import os, shutil, subprocess, glob, csv, time
from pathlib import Path

num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
start_at = 1 #@param {type:"integer"}
stop_after = 0 #@param {type:"integer"}
#@markdown `stop_after=0` runs through the end. Completed jobs are skipped when `on_existing=skip`.

num_recycles=max(1,int(num_recycles)); num_diffusion_samples=max(1,int(num_diffusion_samples))

def detect_device():
  try:
    out=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
    caps=[float(x) for x in out.stdout.split() if x.strip()]
    if caps: return 'gpu',min(caps)
  except Exception: pass
  return 'cpu',None

device,cap=detect_device(); nojit=False; xla_flags=[]
if device=='cpu': flash_impl='xla'; nojit=True
elif cap < 8.0: flash_impl='xla'; xla_flags=['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
elif 8.0 < cap < 9.0: flash_impl='xla'; xla_flags=['--xla_gpu_enable_triton_gemm=false']
else: flash_impl='triton'; xla_flags=['--xla_gpu_enable_triton_gemm=false']
cur=os.environ.get('XLA_FLAGS','')
for flag in xla_flags:
  if flag not in cur: cur=(cur+' '+flag).strip()
if cur: os.environ['XLA_FLAGS']=cur

if os.path.isdir('af3_native_weights') and glob.glob('af3_native_weights/*.bin*'):
  model_dir,use_of3='af3_native_weights',False
else:
  model_dir,use_of3='af3_converted_weights',True
print('Weights:', 'OpenFold3' if use_of3 else 'official AlphaFold 3', '| device:',device,cap)

selected=fold_inputs[max(0,start_at-1):]
if stop_after > 0: selected=selected[:stop_after]
progress_path=Path(OUTPUT_DIR)/'batch_progress.csv'
if not progress_path.exists():
  with progress_path.open('w',newline='') as f: csv.writer(f).writerow(['job','status','seconds','message'])

summary={'completed':0,'skipped':0,'failed':0}
for pos,job in enumerate(selected,start=max(1,start_at)):
  name=job['name']; json_path=f'{INPUT_DIR}/{name}.json'; job_dir=Path(OUTPUT_DIR)/name
  have_results=job_dir.is_dir() and any(job_dir.glob('*.cif'))
  if on_existing=='skip' and have_results:
    print(f'[{pos}/{len(fold_inputs)}] SKIP {name}')
    summary['skipped']+=1; continue
  shutil.rmtree(job_dir,ignore_errors=True)
  cmd=['python','run_alphafold.py',f'--json_path={json_path}',f'--model_dir={model_dir}',
       '--norun_data_pipeline',f'--output_dir={OUTPUT_DIR}','--cache_dir=/tmp/af3_cache','--force_output_dir',
       f'--flash_attention_implementation={flash_impl}',f'--num_recycles={num_recycles}',
       f'--num_diffusion_samples={num_diffusion_samples}']
  if nojit: cmd.append('--nojit')
  if use_of3: cmd.append('--of3_weights')
  print(f'[{pos}/{len(fold_inputs)}] RUN  {name}',flush=True); t=time.time()
  proc=subprocess.run(cmd,text=True); elapsed=round(time.time()-t,1)
  ok=proc.returncode==0 and job_dir.is_dir() and any(job_dir.glob('*.cif'))
  status='completed' if ok else 'failed'; summary[status]+=1
  with progress_path.open('a',newline='') as f:
    csv.writer(f).writerow([name,status,elapsed,'' if ok else f'exit {proc.returncode}'])
  print(f'[{pos}/{len(fold_inputs)}] {status.upper()} {name} ({elapsed}s)',flush=True)

print('Batch summary:',summary)
print('Progress log:',progress_path)


In [ ]:
#@title Display structures + PAE (py2Dmol)
import csv, glob, os, json
import numpy as np
import py2Dmol

load_as_frames = True #@param {type:"boolean"}
viewer_size = 400
#@markdown All predicted models load together, best first (**rank_1, rank_2, ...**), each with its own PAE.
#@markdown - `load_as_frames` **off** -> pick a model from the dropdown.
#@markdown - `load_as_frames` **on**  -> models become frames you can play through (press play / drag the slider).
#@markdown - The interactive PAE matrix sits beside the structure; click or drag-box on it to highlight residues.

# All models in rank order (best first): from the ranking CSV, fall back to globbing.
def collect_models():
  ranking_csv = f'{job_dir}/{jobname}_ranking_scores.csv'
  cifs = []
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for r in csv.DictReader(f):
        rows.append((float(r['ranking_score']), int(r['seed']), int(r['sample'])))
    for _, seed, sample in sorted(rows, reverse=True):
      d = f'{job_dir}/seed-{seed}_sample-{sample}'
      hit = sorted(glob.glob(f'{d}/*_model.cif')) or sorted(glob.glob(f'{d}/*.cif'))
      if hit:
        cifs.append(hit[0])
  if not cifs:
    cifs = (sorted(glob.glob(f'{job_dir}/**/*_model.cif', recursive=True))
            or sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True)))
  return cifs

# Per-model PAE: confidences.json next to the CIF, else the top-level one.
def load_pae(cif):
  d = os.path.dirname(cif)
  cands = [p for p in glob.glob(f'{d}/*_confidences.json')
           if 'summary' not in os.path.basename(p)]
  if not cands:
    top = f'{job_dir}/{jobname}_confidences.json'
    cands = [top] if os.path.exists(top) else []
  if cands:
    pae = json.load(open(cands[0])).get('pae')
    if pae is not None:
      return np.asarray(pae, dtype=float)
  return None

cifs = collect_models()
if not cifs:
  raise FileNotFoundError(f'No model CIFs found in {job_dir}/')
print(f'Loaded {len(cifs)} model(s) from {job_dir}/'
      + ('  (frames - press play)' if load_as_frames else '  (use the dropdown to switch)'))

viewer = py2Dmol.view(size=(viewer_size, viewer_size),
                      pae=True, autoplay=load_as_frames)
for i, cif in enumerate(cifs, start=1):
  pae = load_pae(cif)
  if load_as_frames:
    viewer.add_pdb(cif, name='models', paes=pae)      # same name -> frames (play through)
  else:
    viewer.add_pdb(cif, name=f'rank_{i}', paes=pae)    # distinct names -> dropdown of objects
viewer.show()


In [ ]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


In [ ]:
#@title Download results
from google.colab import files
import os

results_zip = f'{jobname}.result.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}/{jobname}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start**
1. Fill in the sequence(s) in the **Input sequence(s)** cell.
2. Press **Runtime → Run all**.
3. The install cell (first run only) downloads OpenFold3 weights and builds AF3 data files in the background — subsequent runs reuse them.

---

## Model weights

By default the notebook uses the open **OpenFold3** weights (Apache-2.0, no restrictions). To run the **official AlphaFold 3** weights instead, tick the **use_af3_weights** box in the install cell. The weights are now publicly downloadable (no login, key, or DeepMind approval required) and are fetched directly into `af3_native_weights/`, then used automatically (the run cell detects them and drops the `--of3_weights` flag). Leaving **use_af3_weights** off keeps the OpenFold3 path.

Official AlphaFold 3 parameters are subject to the [AF3 weights terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md); run_alphafold prints a reminder of these terms when they are used.

---

## Sequence input

Each molecule type has its own box. Within a box, separate multiple chains with `:`.

| Box | What goes in it | Example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Chains are assigned IDs A, B, C, … following AlphaFold 3's canonical order (protein → RNA → DNA → ligand; CCD ligands before SMILES ligands). Mix freely across boxes to build a complex — e.g. a protein in **protein**, `AUGCAUGC` in **rna**, and `ATP` in **ligand_ccd**.

- **Homo-oligomers**: identical protein sequences are merged automatically, so `SEQ:SEQ` = homodimer, `SEQ:SEQ:SEQ` = homotrimer.
- Protein / DNA / RNA sequences and CCD codes are upper-cased automatically; **SMILES are left exactly as typed** (case is meaningful in SMILES).
- Spaces and newlines inside an entry are ignored, and **extra colons are forgiven** — `SEQ1::::SEQ2` is the same as `SEQ1:SEQ2`. Leave a box empty if unused.
- *Note:* because `:` separates entries, an atom-mapped SMILES that itself contains a colon (e.g. `[C:1]`) isn't supported via the box — use a raw AF3 JSON for that edge case.

## Seeds

Enter one or more model seeds in the **seeds** box, comma-separated (e.g. `1,2,3`). Each seed is an independent prediction (more seeds = more sampling, more runtime). Non-numeric characters are ignored and duplicates are dropped, so `1, 1, foo, 7` becomes seeds `1` and `7`.

## MSA modes

- **`mmseqs2_server`** *(recommended)*: queries the public [ColabFold](https://colabfold.mmseqs.com/) MMseqs2 API. Covers UniRef30 + environmental sequences for proteins. RNA/DNA chains always run MSA-free (ColabFold is protein-only).
- **`single_sequence`**: no MSA, query sequence only. Faster but less accurate, especially for monomers with close homologs.

## Output files (inside the downloaded zip)

| File | Contents |
|---|---|
| `*.cif` | Best-ranked structure in mmCIF format. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probs. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Ranking scores for all seed × sample combinations. |
| `seed-N_sample-M/` | Individual prediction directories (one per seed/sample). |
| `TERMS_OF_USE.md` | Apache 2.0 notice (OpenFold3 weights have no commercial restrictions). |

## Interpreting confidence scores

- **pLDDT > 90**: very high confidence.
- **pLDDT 70–90**: confident, backbone generally reliable.
- **pLDDT 50–70**: low confidence, treat with caution.
- **pLDDT < 50**: very low, likely disordered or incorrect.
- **PAE**: lower values = confident relative positioning between residue pairs. Useful for assessing interface quality in complexes.
- **ipTM > 0.8**: strong evidence for a well-defined complex interface. **ipTM is `n/a` for single-chain jobs** (there is no interface to score).

## Troubleshooting

- **Check runtime type**: `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
- **OOM error**: reduce sequence length or use a larger-memory GPU runtime.
- **MSA server timeout**: the public ColabFold server is rate-limited. Try again later or switch to `single_sequence` mode.
- **Download popup blocked**: disable your ad blocker.
- **Weight download slow**: the OF3 checkpoint is ~1.4 GB from AWS S3; the install cell waits automatically.

## License

AlphaFold 3 source code and OpenFold3 weights are both licensed under [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Outputs generated with OpenFold3 weights are **not** subject to Google DeepMind's AlphaFold 3 Output Terms of Use and may be used freely, including commercially.

## Bugs / feedback

Report issues at https://github.com/sokrypton/alphafold3/issues